In [3]:
!pip install "langchain==0.3.27" "langchain-community==0.3.27" "langchain-text-splitters==0.3.11" "langchain-experimental==0.3.4" "langchain-core==0.3.78" sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.1
    Uninstalling packaging-26.1:
      Successfully uninstalled packaging-26.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you have langchain-core 0.3.78 which is incompatibl

In [1]:
# fix_5_fallback.py
import pandas as pd
import numpy as np
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_experimental.text_splitter import SemanticChunker


In [2]:
# Build vector store (reuse from Fix #4)
df = pd.read_csv("/content/customer_support_tickets.csv")
documents = df["Ticket Description"].dropna().tolist()

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
splitter   = SemanticChunker(embeddings=embeddings)
chunks = []
for doc in documents[:300]:
    chunks.extend(splitter.split_text(doc))

vectorstore = FAISS.from_texts(chunks, embeddings)


/tmp/ipykernel_1819/3846664824.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warn

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
# Confidence-gated retrieval function
def rag_with_fallback(query: str, similarity_threshold: float = 0.70) -> dict:
    """
    Retrieves documents and gates on similarity score.
    If the best match is below threshold, returns a fallback instead of
    passing empty/low-quality context to the LLM.

    FAISS with inner-product (IP) index returns similarity scores in [0, 1]
    when using normalized embeddings. Adjust threshold for your vector store.
    """
    results = vectorstore.similarity_search_with_score(query, k=5)

    # Filter results above confidence threshold
    # Note: FAISS IP scores/higher is more similar
    confident = [(doc, score) for doc, score in results if score >= similarity_threshold]

    if not confident:
        return {
            "answer"      : (
                "I don't have reliable information in my knowledge base to answer "
                "this accurately. Please contact our support team directly at "
                "support@company.com or call 1-800-XXX-XXXX."
            ),
            "source"      : "fallback",
            "confidence"  : "low",
            "best_score"  : round(float(results[0][1]), 4) if results else None
        }

    context = "\n\n".join([doc.page_content for doc, _ in confident[:3]])

    # In production: replace this print with the actual LLM call
    # response = llm_chain.run(query=query, context=context)
    return {
        "answer"      : f"[LLM RESPONSE USING CONTEXT]\n\nContext used:\n{context[:400]}...",
        "source"      : "rag",
        "confidence"  : "high",
        "chunks_used" : len(confident),
        "best_score"  : round(float(confident[0][1]), 4)
    }


In [4]:
# Test with in-domain and out-of-domain queries
test_queries = [
    "What is the refund policy for duplicate charges?",         # in-domain → should pass
    "How do I reset my account password?",                      # in-domain → should pass
    "What is the melting point of tungsten carbide alloys?",    # out-of-domain → should fallback
    "Explain the history of the Byzantine Empire in detail.",   # out-of-domain → should fallback
]

for query in test_queries:
    result = rag_with_fallback(query, similarity_threshold=0.70)
    print(f"\nQuery   : {query}")
    print(f"Source  : {result['source'].upper()}")
    print(f"Score   : {result.get('best_score', 'N/A')}")
    print(f"Answer  : {result['answer'][:150]}...")
    print("-" * 70)


Query   : What is the refund policy for duplicate charges?
Source  : FALLBACK
Score   : 0.5058
Answer  : I don't have reliable information in my knowledge base to answer this accurately. Please contact our support team directly at support@company.com or c...
----------------------------------------------------------------------

Query   : How do I reset my account password?
Source  : FALLBACK
Score   : 0.3689
Answer  : I don't have reliable information in my knowledge base to answer this accurately. Please contact our support team directly at support@company.com or c...
----------------------------------------------------------------------

Query   : What is the melting point of tungsten carbide alloys?
Source  : RAG
Score   : 1.0201
Answer  : [LLM RESPONSE USING CONTEXT]

Context used:
Please send me a PM. I'm taking the following instructions to the company. Go to I've noticed a peculiar e...
----------------------------------------------------------------------

Query   : Explain t